[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Virtual Environments &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell sets up the project's folder, `run` and `pip`, and the cell after it writes the
tutorial's script. Run them first, then the tasks in order, since every task uses the environment the
tasks before it made. The last cell removes the scratch folder.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT = Path("scratch/stations")
PROJECT.mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun
os.environ["COLUMNS"] = "80"                  # and print reports 80 characters wide


def run(*command, folder=PROJECT):
    """Run a command in a folder, and return its exit code and what it printed, less this computer's paths."""
    finished = subprocess.run([str(part) for part in command], cwd=folder, capture_output=True, text=True)
    printed = (finished.stdout + finished.stderr).replace(f"{Path(folder).resolve()}/", "")
    return finished.returncode, re.sub(r" in \d+\.\d+s\b", "", printed).rstrip()


def pip(environment, *arguments):
    """Run this notebook's pip for the Python in an environment: python -m pip --python ENVIRONMENT ..."""
    return run(sys.executable, "-m", "pip", "--python", environment, "--disable-pip-version-check", *arguments)


print("ready:", PROJECT)


ready: scratch/stations


In [2]:
%%writefile scratch/stations/tutorial.py
"""A tutorial from 2021: the release tags of a package, in order."""

from packaging.version import parse

tags = ["2.10", "2.9", "2.10rc1", "latest"]
print(sorted(tags, key=parse))


Writing scratch/stations/tutorial.py


**1.** An environment, and what is in it.


In [3]:
code, printed = run(sys.executable, "-m", "venv", "--without-pip", "task-env")
print("exit code:", code)

print(sorted(path.name for path in (PROJECT / "task-env").iterdir()))
config = (PROJECT / "task-env" / "pyvenv.cfg").read_text()
print([line.split(" = ")[0] for line in config.splitlines()])


exit code: 0
['.gitignore', 'bin', 'include', 'lib', 'pyvenv.cfg']
['home', 'include-system-site-packages', 'version', 'executable', 'command']


On Python 3.12 the folder has no `.gitignore`, which Python 3.13 and later write, and everything else
is the same.


**2.** Inside an environment?


In [4]:
check = "import sys; print('inside an environment:', sys.prefix != sys.base_prefix)"
code, printed = run("task-env/bin/python", "-c", check)
print(printed)


inside an environment: True


The environment's `python` was run by its path, and nothing was activated.


**3.** Packages in, and freeze.


In [5]:
code, printed = pip("task-env", "install", "-q", "packaging==21.3", "pyparsing==3.3.2")
print("exit code:", code)

code, printed = pip("task-env", "freeze")
print(printed)


exit code: 0
packaging==21.3
pyparsing==3.3.2


`--python task-env`, inside `pip`, sent the notebook's pip to the environment's Python, which has no
pip of its own.


**4.** The tutorial, in two Pythons.


In [6]:
for python in ["task-env/bin/python", sys.executable]:
    code, printed = run(python, "tutorial.py")
    name = "the notebook's Python" if python == sys.executable else python
    print(f"{name:<22} exit code {code}: {printed.splitlines()[-1]}")


task-env/bin/python    exit code 0: ['latest', '2.9', '2.10rc1', '2.10']
the notebook's Python  exit code 1: packaging.version.InvalidVersion: Invalid version: 'latest'


The environment has `packaging` 21.3, which accepted `latest`. The notebook's Python has a later
release, which raises `InvalidVersion`.


**5.** An environment that sees the packages around it.


In [7]:
run(sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", "task-shared")

for environment in ["task-env", "task-shared"]:
    code, printed = run(f"{environment}/bin/python", "-c", "import pytest")
    print(f"{environment:<12} import pytest works: {code == 0}")


task-env     import pytest works: False
task-shared  import pytest works: True


`task-shared` sees the notebook's Python's pytest, and `task-env` sees only what was installed into
it, which is not pytest.


**6.** Removed, and made again.


In [8]:
shutil.rmtree(PROJECT / "task-env")
run(sys.executable, "-m", "venv", "--without-pip", "task-env")

code, printed = pip("task-env", "freeze")
print("freeze lists:", repr(printed))


freeze lists: ''


An empty string: a new environment holds nothing, which is why a record of what an environment held,
the subject of the **Requirements and Pinning** notebook, is what makes it cheap to rebuild.

Last, remove the scratch folder:


In [9]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Virtual Environments](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/07-virtual-environments.ipynb)  &nbsp;&middot;&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)
